Setup Data

In [0]:
%sql
CREATE OR REPLACE TABLE ecommerce_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO ecommerce_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed'),
(107,'Vikram Singh','Delhi','Camera',1,55000,'Placed'),
(108,'Meera Nair','Bangalore','Laptop',1,80000,'Placed');

num_affected_rows,num_inserted_rows
8,8


PART 1 — INSERT Exercises
1. Insert a new order for Rahul Verma from Mumbai buying a Tablet worth 27000.

In [0]:
%sql
insert into ecommerce_orders values(109,"Rahul Varma","Mumbai","Tablet",1,27000,"Placed");

num_affected_rows,num_inserted_rows
1,1


2. Insert two new orders:
Arjun Nair buying Mobile
Kavya Reddy buying Laptop

In [0]:
%sql
insert into ecommerce_orders values(110,"Arjun Nair","Mumbai","Mobile",1,10000,"Placed"),(111,"Kavya Reddy","Bangalore","Laptop",1,90000,"Placed");

num_affected_rows,num_inserted_rows
2,2


3. Insert an order with status Shipped.

In [0]:
%sql
insert into ecommerce_orders values(112,"Rahul Chaudry","Mumbai","Telivision",1,190000,"Shipped");

num_affected_rows,num_inserted_rows
1,1


4. Insert an order with quantity 5 headphones.

In [0]:
%sql
insert into ecommerce_orders values(113,"Priya Reddy","Bangalore","HeadPhones",5,75000,"Shipped")

num_affected_rows,num_inserted_rows
1,1


5. Insert three new orders for Hyderabad customers.

In [0]:
%sql
insert into ecommerce_orders values(114,"Nikil Varma","Hyderabad","Camera",1,56000,"Placed"),(115,"Santhosh","Hyderabad","Fridge",1,780000,"Shipped"),(116,"Nikitha","Hyderabad","Laptop",1,75000,"Placed");

num_affected_rows,num_inserted_rows
3,3


PART 2 — UPDATE Exercises

1. Change order_status of order 101 to Shipped.

In [0]:
%sql
update ecommerce_orders set order_status="Shipped" where order_id=101;

num_affected_rows
1


2. Increase quantity of order 102 by 1.

In [0]:
%sql
update ecommerce_orders set quantity=quantity+1 where order_id=102;

num_affected_rows
1


3. Change price of all Laptop orders to 78000.

In [0]:
%sql
update ecommerce_orders set price=78000 where product="Laptop"

num_affected_rows
5


4. Update city of Amit Sharma to Secunderabad.

In [0]:
%sql
update ecommerce_orders set city="Secunderabad" where customer_name="Amit Sharma"

num_affected_rows
1


5. Change order_status to Delivered where product = Mobile.

In [0]:
%sql
update ecommerce_orders set order_status="Delivered" where product="Mobile"

num_affected_rows
3


6. Update price of Headphones to 2500.

In [0]:
%sql
update ecommerce_orders set price=2500 where product="Headphones"

num_affected_rows
2


7. Increase price of all Tablet products by 1000.

In [0]:
%sql
update ecommerce_orders set price=price+1000 where product="Tablet"

num_affected_rows
2


8. Change order_status of orders from Bangalore to Processing.

In [0]:
%sql
update ecommerce_orders set order_status="Processing" where city="Bangalore"

num_affected_rows
4


9. Update quantity to 2 where quantity = 1.

In [0]:
%sql
update ecommerce_orders set quantity=1 where quantity=2

num_affected_rows
1


10. Change city Ahmedabad to Surat.

In [0]:
%sql
update ecommerce_orders set city="Surat" where city="Ahmedabad"

num_affected_rows
1


PART 3 — DELETE Exercises

1. Delete orders where order_status = Cancelled.

In [0]:
%sql
delete  from ecommerce_orders where order_status="Cancelled"

num_affected_rows
1


2. Delete orders where quantity > 3.

In [0]:
%sql
Delete from ecommerce_orders where quantity>3

num_affected_rows
1


3. Delete orders where product = Camera.

In [0]:
%sql
Delete from ecommerce_orders where product="Camera"

num_affected_rows
2


In [0]:
%sql
Delete from ecommerce_orders where city="Kolkata"

num_affected_rows
1


In [0]:
%sql
Delete from ecommerce_orders where price<5000

num_affected_rows
1


In [0]:
%sql
Delete from ecommerce_orders where customer_name Like"A%"

num_affected_rows
2


In [0]:
%sql
Delete from ecommerce_orders where product="Tablet"

num_affected_rows
1


In [0]:
%sql
Delete from ecommerce_orders where city="Mumbai" and quantity=1

num_affected_rows
1


9. Delete orders with price greater than 80000.

In [0]:
%sql
Delete from ecommerce_orders where price>80000

num_affected_rows
1


In [0]:
%sql
DELETE FROM ecommerce_orders 
WHERE order_id = (SELECT MAX(order_id) FROM ecommerce_orders);


num_affected_rows
1


In [0]:
%sql
select * from ecommerce_orders;

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,3,30000,Processing
108,Meera Nair,Bangalore,Laptop,1,78000,Processing
111,Kavya Reddy,Bangalore,Laptop,1,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,78000,Placed


PART 4 — MERGE / UPSERT Exercises

Create incoming dataset.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed'),
(111,'Farhan Ali','Hyderabad','Laptop',1,79000,'Placed')
AS incoming_orders(order_id,customer_name,city,product,quantity,price,order_status);

1. Merge incoming_orders into ecommerce_orders.
2. Update matching orders using WHEN MATCHED.
3. Insert new orders using WHEN NOT MATCHED.

In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
target.customer_name = source.customer_name,
target.city = source.city,
target.product = source.product,
target.quantity = source.quantity,
target.price = source.price,
target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,3,0,2


4. After merge, list orders sorted by order_id.

In [0]:
%sql
select * from ecommerce_orders order by order_id

order_id,customer_name,city,product,quantity,price,order_status
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
108,Meera Nair,Bangalore,Laptop,1,78000,Processing
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed


5. Identify which rows were inserted vs updated.

In [0]:
%sql
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED THEN
UPDATE SET order_status = s.order_status

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


In [0]:
%sql
SELECT * FROM ecommerce_orders;

order_id,customer_name,city,product,quantity,price,order_status
108,Meera Nair,Bangalore,Laptop,1,78000,Processing
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
110,Divya Menon,Kochi,Mobile,1,27000,Placed
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed


6.Update ONLY order_status

In [0]:
%sql
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED THEN
UPDATE SET t.order_status = s.order_status

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


Exercise 7
Update quantity ONLY when it increase

In [0]:
%sql
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.quantity > t.quantity THEN
UPDATE SET t.quantity = s.quantity

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


Exercise 8
Ignore Cancelled Orders

In [0]:
%sql
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.order_status != 'Cancelled' THEN
UPDATE SET *

WHEN NOT MATCHED AND s.order_status != 'Cancelled' THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


Exercise 9
Insert ONLY Laptop Products

In [0]:
%sql
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED AND s.product = 'Laptop' THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
5,5,0,0


Exercise 10
Update Price ONLY if Highe

In [0]:
%sql
MERGE INTO ecommerce_orders t
USING incoming_orders s
ON t.order_id = s.order_id

WHEN MATCHED AND s.price > t.price THEN
UPDATE SET t.price = s.price

WHEN NOT MATCHED THEN
INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


PART 5 — Realistic UPSERT Scenario

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS
SELECT * FROM VALUES
(103,'Rohit Mehta','Mumbai','Headphones',5,2200,'Delivered'),
(106,'Ananya Das','Kolkata','Mobile',2,28000,'Shipped'),
(112,'Sanjay Gupta','Delhi','Tablet',1,24000,'Placed')
AS daily_updates(order_id,customer_name,city,product,quantity,price,order_status);

Exercises

1. Merge daily_updates into ecommerce_orders.
2. Update quantity for existing orders.
3. Insert orders that do not exist.

In [0]:
%sql
MERGE INTO ecommerce_orders AS target
USING daily_updates AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
  target.quantity = source.quantity,
  target.price = source.price,
  target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT (
  order_id,
  customer_name,
  city,
  product,
  quantity,
  price,
  order_status
)
VALUES (
  source.order_id,
  source.customer_name,
  source.city,
  source.product,
  source.quantity,
  source.price,
  source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,0,0,3


Exercise 4 — Count Updated Rows

In [0]:
%sql
SELECT COUNT(*) AS updated_rows
FROM daily_updates d
JOIN ecommerce_orders e
ON d.order_id = e.order_id;

updated_rows
3


Exercise 5 — Count Inserted Rows

In [0]:
%sql
SELECT COUNT(*) AS inserted_rows
FROM daily_updates d
LEFT ANTI JOIN ecommerce_orders e
ON d.order_id = e.order_id;

inserted_rows
0


 Exercise 6 — Final Table Sorted by Price

In [0]:
%sql
SELECT *
FROM ecommerce_orders
ORDER BY price DESC;

order_id,customer_name,city,product,quantity,price,order_status
111,Farhan Ali,Hyderabad,Laptop,1,79000,Placed
108,Meera Nair,Bangalore,Laptop,1,78000,Processing
104,Sneha Iyer,Chennai,Laptop,1,72000,Shipped
102,Priya Reddy,Bangalore,Mobile,4,30000,Delivered
106,Ananya Das,Kolkata,Mobile,2,28000,Shipped
110,Divya Menon,Kochi,Mobile,1,27000,Placed
109,Rahul Verma,Mumbai,Tablet,2,26000,Placed
112,Sanjay Gupta,Delhi,Tablet,1,24000,Placed
103,Rohit Mehta,Mumbai,Headphones,5,2200,Delivered
